In [ ]:
!pip install pandas psycopg2-binary transformers scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 32.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Fungsi untuk membaca kolom dari file Excel
def read_excel_columns(file_path, columns_to_compare=None):
    df = pd.read_excel(file_path)

    if columns_to_compare:
        # Hanya ambil kolom-kolom yang ingin dibandingkan
        df = df[columns_to_compare]

    return list(df.columns)

# Fungsi untuk menghitung embedding dari kata atau frasa menggunakan IndoBERT
def get_embeddings(text_list, model, tokenizer):
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk menghitung cosine similarity antar embedding
def compute_similarity(embedding1, embedding2):
    embedding1 = embedding1.detach().numpy()
    embedding2 = embedding2.detach().numpy()
    return cosine_similarity(embedding1, embedding2)[0][0]

# Membaca skema dari dua file Excel, lalu membandingkannya
def compare_schemas(excel_file_1, excel_file_2, model, tokenizer, columns_to_compare_1=None, columns_to_compare_2=None):
    # Baca kolom dari kedua file Excel (hanya kolom tertentu jika diberikan)
    excel1_columns = read_excel_columns(excel_file_1, columns_to_compare_1)
    excel2_columns = read_excel_columns(excel_file_2, columns_to_compare_2)

    # Ambil embedding untuk nama-nama kolom dari kedua file Excel
    excel1_embeddings = get_embeddings(excel1_columns, model, tokenizer)
    excel2_embeddings = get_embeddings(excel2_columns, model, tokenizer)

    # Membandingkan tiap kolom dari Excel1 dengan tiap kolom dari Excel2
    similarity_results = {}
    for i, excel1_col in enumerate(excel1_columns):
        best_match = None
        best_score = 0
        for j, excel2_col in enumerate(excel2_columns):
            similarity = compute_similarity(excel1_embeddings[i].unsqueeze(0), excel2_embeddings[j].unsqueeze(0))
            if similarity > best_score:
                best_match = excel2_col
                best_score = similarity

        similarity_results[excel1_col] = (best_match, best_score)

    return similarity_results

# Main program
if __name__ == "__main__":
    # File Excel yang ingin dibandingkan
    excel_file_1 = "skema1.xlsx"  # ganti dengan path file Excel 1 Anda
    excel_file_2 = "Skema2.xlsx"  # ganti dengan path file Excel 2 Anda

    # Daftar kolom yang ingin dibandingkan dari kedua file
    columns_to_compare_1 = ['Variabel', 'Nama Variabel']  # Ganti dengan nama kolom dari Excel 1
    columns_to_compare_2 = ['Kode Variabel', 'Deskripsi Variabel']  # Ganti dengan nama kolom dari Excel 2

    # Load IndoBERT dari Hugging Face
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Lakukan perbandingan skema
    similarity_results = compare_schemas(excel_file_1, excel_file_2, model, tokenizer, columns_to_compare_1, columns_to_compare_2)

    # Tampilkan hasil perbandingan
    for excel1_col, (excel2_col, score) in similarity_results.items():
        print(f"Kolom '{excel1_col}' dari Excel1 cocok dengan '{excel2_col}' dari Excel2 dengan skor similarity: {score:.4f}")


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Kolom 'Variabel' dari Excel1 cocok dengan 'Deskripsi Variabel' dari Excel2 dengan skor similarity: 0.8472
Kolom 'Nama Variabel' dari Excel1 cocok dengan 'Kode Variabel' dari Excel2 dengan skor similarity: 0.9066


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Fungsi untuk membaca kolom dan baris dari file Excel
def read_excel_columns_and_rows(file_path, columns_to_compare=None):
    df = pd.read_excel(file_path)

    if columns_to_compare:
        # Hanya ambil kolom-kolom yang ingin dibandingkan
        df = df[columns_to_compare]

    return df

# Fungsi untuk menghitung embedding dari kata atau frasa menggunakan IndoBERT
def get_embeddings(text_list, model, tokenizer):
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk menghitung cosine similarity antar embedding
def compute_similarity(embedding1, embedding2):
    embedding1 = embedding1.detach().numpy()
    embedding2 = embedding2.detach().numpy()
    return cosine_similarity(embedding1, embedding2)[0][0]

# Membaca skema dari dua file Excel, lalu membandingkannya
def compare_schemas_and_rows(excel_file_1, excel_file_2, model, tokenizer, columns_to_compare_1=None, columns_to_compare_2=None):
    # Baca kolom dan baris dari kedua file Excel (hanya kolom tertentu jika diberikan)
    excel1_df = read_excel_columns_and_rows(excel_file_1, columns_to_compare_1)
    excel2_df = read_excel_columns_and_rows(excel_file_2, columns_to_compare_2)

    # Ambil nama kolom
    excel1_columns = list(excel1_df.columns)
    excel2_columns = list(excel2_df.columns)

    # Ambil embedding untuk nama-nama kolom dari kedua file Excel
    excel1_embeddings = get_embeddings(excel1_columns, model, tokenizer)
    excel2_embeddings = get_embeddings(excel2_columns, model, tokenizer)

    # Membandingkan tiap kolom dari Excel1 dengan tiap kolom dari Excel2
    similarity_results = {}
    for i, excel1_col in enumerate(excel1_columns):
        best_match = None
        best_score = 0
        for j, excel2_col in enumerate(excel2_columns):
            similarity = compute_similarity(excel1_embeddings[i].unsqueeze(0), excel2_embeddings[j].unsqueeze(0))
            if similarity > best_score:
                best_match = excel2_col
                best_score = similarity

        similarity_results[excel1_col] = (best_match, best_score)

    # Mencari baris yang berbeda
    row_differences = []
    for index, (row1, row2) in enumerate(zip(excel1_df.itertuples(index=False), excel2_df.itertuples(index=False))):
        if list(row1) != list(row2):
            row_differences.append(index + 2)  # +2 untuk menyesuaikan nomor baris (karena indeks mulai dari 0, dan +1 untuk header)

    return similarity_results, row_differences

# Main program
if __name__ == "__main__":
   # File Excel yang ingin dibandingkan
    excel_file_1 = "skema1.xlsx"  # ganti dengan path file Excel 1 Anda
    excel_file_2 = "Skema2.xlsx"  # ganti dengan path file Excel 2 Anda

    # Daftar kolom yang ingin dibandingkan dari kedua file
    columns_to_compare_1 = ['Variabel', 'Nama Variabel']  # Ganti dengan nama kolom dari Excel 1
    columns_to_compare_2 = ['Kode Variabel', 'Deskripsi Variabel']  # Ganti dengan nama kolom dari Excel 2

    # Load IndoBERT dari Hugging Face
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Lakukan perbandingan skema dan baris
    similarity_results, row_differences = compare_schemas_and_rows(excel_file_1, excel_file_2, model, tokenizer, columns_to_compare_1, columns_to_compare_2)

    # Tampilkan hasil perbandingan kolom
    print("Perbandingan Kolom:")
    for excel1_col, (excel2_col, score) in similarity_results.items():
        print(f"Kolom '{excel1_col}' dari Excel1 cocok dengan '{excel2_col}' dari Excel2 dengan skor similarity: {score:.4f}")

    # Tampilkan baris yang berbeda
    print("\nBaris yang berbeda:")
    if row_differences:
        print("Baris yang berbeda ditemukan pada baris:", row_differences)
    else:
        print("Tidak ada perbedaan baris yang ditemukan.")


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Perbandingan Kolom:
Kolom 'Variabel' dari Excel1 cocok dengan 'Deskripsi Variabel' dari Excel2 dengan skor similarity: 0.8472
Kolom 'Nama Variabel' dari Excel1 cocok dengan 'Kode Variabel' dari Excel2 dengan skor similarity: 0.9066

Baris yang berbeda:
Baris yang berbeda ditemukan pada baris: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 1

In [ ]:
!pip install pandas sentence-transformers scikit-learn openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 15.3 MB/s eta 0:00:00


In [ ]:
!pip install ace_tools

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Langkah 1: Membaca file Excel
file1 = 'skema1.xlsx'  # Nama file pertama
file2 = 'skema2.xlsx'  # Nama file kedua

# Membaca kolom-kolom dari file Excel
df1 = pd.read_excel(file1, usecols=['Variabel', 'Nama Variabel'])  # Ganti 'kolom1', 'kolom2' dengan kolom yang relevan
df2 = pd.read_excel(file2, usecols=['Kode Variabel', 'Deskripsi Variabel'])  # Ganti 'kolomA', 'kolomB' dengan kolom yang relevan

# Langkah 2: Gabungkan kolom yang ingin dibandingkan
# Misal kita ingin membandingkan dua kolom dari df1 dengan dua kolom dari df2
#text1 = df1['Variabel'] + ' ' + df1['Nama Variabel'].str.lower()  # Gabungkan isi dua kolom di file1
#text2 = df2['Kode Variabel'] + ' ' + df2['Deskripsi Variabel'].str.lower()  # Gabungkan isi dua kolom di file2
text1 =  df1['Nama Variabel'].str.lower()  # Gabungkan isi dua kolom di file1
text2 =  df2['Deskripsi Variabel'].str.lower()  # Gabungkan isi dua kolom di file2


# Langkah 3: Load model pre-trained Sentence-BERT
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')  # Model untuk mendapatkan embedding semantik

# Langkah 4: Dapatkan embedding untuk setiap baris
embeddings1 = model.encode(text1.tolist())  # Mengubah teks file1 menjadi embedding
embeddings2 = model.encode(text2.tolist())  # Mengubah teks file2 menjadi embedding

# Langkah 5: Hitung kesamaan kosinus antara setiap baris dari dua file
similarity_matrix = cosine_similarity(embeddings1, embeddings2)

# Langkah 6: Simpan hasil dalam format no, teks1, teks2, dan nilai kesamaan
results = []
for idx1, similarities in enumerate(similarity_matrix):
    idx2 = similarities.argmax()  # Indeks baris di df2 yang paling mirip dengan baris idx1 di df1
    similarity_score = similarities[idx2]
    results.append([idx1 + 1, text1[idx1], text2[idx2], similarity_score])

# Buat DataFrame dari hasil
df_results = pd.DataFrame(results, columns=['No', 'Teks1', 'Teks2', 'Nilai Kesamaan'])

# Menyimpan hasil ke file Excel
df_results.to_csv('hasil_kesamaan_berformat3.csv', index=False)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
